**NOTICE:**  
The U.S. Army Corps of Engineers, Risk Management Center (USACE-RMC) makes no guarantees about the results, or appropriateness of outputs, obtained from Numerics.


# 08. Optimization
This notebook covers local, global, and constrained optimization methods in Numerics.

## What You'll Learn
- Local optimization (Nelder-Mead, BFGS, Powell)
- Global optimization (DE, MS, MLSL, Particle Swarm, SCE, Simulated Annealing)
- Constrained optimization (Augmented Lagrange)
- Practical applications

## What is Optimization?
An optimization problem seeks to find:

```math
\min_{\mathbf{x}} f(\mathbf{x})
```

subject to:

```math
\mathbf{x}_L \leq \mathbf{x} \leq \mathbf{x}_U
```

where $f(\mathbf{x})$ is the objective function, $\mathbf{x}$ is the parameter vector, and $\mathbf{x}_L$, $\mathbf{x}_U$ are lower and upper bounds.

For maximization problems, minimize $-f(\mathbf{x})$ or use the `Maximize()` method.

## When to Use Optimization vs MCMC

Use Optimization when:
- You need point estimatest only (no uncertainty)
- Computational speed is critical
- You have a well-defined objective function

Use MCMC when:
- You need uncertainty quantification
- You want full posterior distributions
- You have prior information to incorporate

## Setup


In [ ]:
import pythonnet
pythonnet.load("coreclr")

import clr
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import animation
from IPython.display import HTML, display
import matplotlib as mpl

# Load Numerics DLL
dll_path = Path(r"C:\GIT\Numerics\Numerics\bin\Debug\net8.0\Numerics.dll")
clr.AddReference(str(dll_path))

from Numerics.Mathematics.Optimization import (NelderMead, BFGS, Powell, ParticleSwarm, SimulatedAnnealing, AugmentedLagrange, 
                                                DifferentialEvolution, MultiStart, LocalMethod, ShuffledComplexEvolution, MLSL, 
                                                ConstraintType, IConstraint, Constraint)
from System import Func, Double, Array
from System.Collections.Generic import List

print("✓ Setup complete")

## Local Optimization

Local optimization methods find the nearest local minimum from a starting point. They are fast and efficient for smooth, unimodal functions but may get trapped in local minima for multimodal problems.

Methods Available
- Adam
- BFGS
- Brent Search
- Golden Selection
- Gradient Descent
- Nelder Mead
- Powell

BFGS is gradient-based and requires derivatives, while Nelder-Mead and Powell are derivative-free. Gradient-based solvers converge fast when the objective is smooth and derivatives are reliable. Derivative-free solvers are more robust when gradients are noisy or unavailable.

### BFGS (Broyden-Fletcher-Goldfarb-Shanno)
BFGS is a quasi-Newton method that builds an approximation to the Hessian matrix using gradient information [[1]](#1). It's one of the most effective methods for smooth, unconstrained optimization.

Advantages: Fast convergence, memory efficient, works well for most smooth problems.        
Disadvantages: Can fail on non-smooth functions, requires bounded search space.

### Nelder-Mead Simplex
The Nelder-Mead method is a direct search algorithm that uses a simplex (geometric figure with $n+1$ vertices in n dimensions) to search the parameter space [[2]](#2). It's robust and doesn't require derivatives.

Advantages: Very robust, doesn't require derivatives, handles non-smooth functions.     
Disadvantages: Slower convergence than gradient-based methods, can stagnate.

### Powell's Method
Powell's method is a conjugate direction algorithm that doesn't require derivatives [[1]](#1). It performs successive line searches along conjugate directions.

Advantages: No derivatives required, good for smooth functions.     
Disadvantages: Can be slow in high dimensions, sensitive to scaling.

## Example: Rosenbrock Function
Rosenbrock is a classic optimization test function: $f(x, y) = (a - x)^2 + b(y - x^2)^2$. With a=1 and b=100, the global minimum is at (1,1) with f=0. 

We will run all three methods on this function to find the minimum.


In [ ]:
# Rosenbrock function
def rosenbrock(params):
    n = len(params)
    F = 0
    for i in range(n-1):
        F += 100 * (params[i + 1] - params[i] * params[i])**2 + (1 - params[i])**2
    return F

# Visualize the function
x = np.linspace(-2, 2, 400)
y = np.linspace(-1, 3, 400)
X, Y = np.meshgrid(x, y)
Z = rosenbrock([X, Y])

fig = plt.figure(figsize=(14, 5))

# Contour plot
ax1 = fig.add_subplot(121)
contour = ax1.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis')
ax1.plot(1, 1, 'r*', markersize=20, label='Global Minimum (1, 1)')
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_title('Rosenbrock Function - Contour Plot', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 3D surface
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8, edgecolor='none')
ax2.scatter([1], [1], [0], color='red', s=100, label='Minimum')
ax2.set_xlabel('x', fontsize=10)
ax2.set_ylabel('y', fontsize=10)
ax2.set_zlabel('f(x, y)', fontsize=10)
ax2.set_title('Rosenbrock Function - 3D Surface', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Global minimum:")
print(f"  Location: (1, 1)")
print(f"  Value:    {rosenbrock([1, 1])}")

# Initialize optimization parameters
initial = Array[Double]([0.0, 0.0])
lower = Array[Double]([-2.048, -2.048])
upper = Array[Double]([2.048, 2.048])

rosenbrock_net = Func[Array[Double], Double](rosenbrock)

# Optimization routines
bfgs = BFGS(rosenbrock_net, 2, initial, lower, upper)
bfgs.Minimize()
bfgs_soln = bfgs.BestParameterSet.Values

nelder = NelderMead(rosenbrock_net, 2, initial, lower, upper)
nelder.Minimize()
nelder_soln = nelder.BestParameterSet.Values

powell = Powell(rosenbrock_net, 2, initial, lower, upper)
powell.Minimize()
powell_soln = powell.BestParameterSet.Values

print('\n'  + " Finding Minimum of Rosenbrock Function:")

table = pd.DataFrame([
    {'Method': 'BFGS',
                'Result': np.round(bfgs_soln, 9),
                'Error': np.round(list(bfgs_soln) - np.array([1, 1]), 9)},
        {'Method': 'Nelder-Mead',
                'Result': np.round(nelder_soln,9),
                'Error': np.round(list(nelder_soln) - np.array([1, 1]), 9)},
        {'Method': 'Powell',
                'Result': np.round(powell_soln, 14),
                'Error': np.round(list(powell_soln) - np.array([1, 1]), 14)}])

display(table)


Now we will look at the path the solver takes when trying to find the minimum of the Rosenbrock function! We will specifically look at the path of the BFGS solver.


In [ ]:
# Use Numerics trace output for BFGS path and objective values
bfgs_trace = BFGS(rosenbrock_net, 2, initial, lower, upper)
bfgs_trace.RecordTraces = True
bfgs_trace.Minimize()

# Extract trace from Numerics
trace_list = list(bfgs_trace.ParameterSetTrace)
trace_params = np.array([[float(ps.Values[0]), float(ps.Values[1])] for ps in trace_list], dtype=float)
trace_values = np.array([float(ps.Fitness) for ps in trace_list], dtype=float)

# Make sure the best parameter set is included in the path
best_params = np.array([float(bfgs_trace.BestParameterSet.Values[0]), float(bfgs_trace.BestParameterSet.Values[1])], dtype=float)
if len(trace_params) == 0 or not np.allclose(trace_params[-1], best_params):
    trace_params = np.vstack([trace_params, best_params])
    trace_values = np.append(trace_values, float(bfgs_trace.BestParameterSet.Fitness))

# Plots!
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(trace_values, color='steelblue', linewidth=1)
axes[0].set_title('BFGS Objective Trace', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Objective value')
axes[0].grid(True, alpha=0.3)

if len(trace_params) > 1:
    axes[1].contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis')
    axes[1].plot(trace_params[:, 0], trace_params[:, 1], color='red', linewidth=1.5, label='BFGS path')
    axes[1].scatter([trace_params[0, 0]], [trace_params[0, 1]], color='white', edgecolor='black', s=50, zorder=5, label='Start')
    # Zorder=10 brings the end point in front of the contour lines
    axes[1].scatter([best_params[0]], [best_params[1]], color='yellow', edgecolor='black', s=50, zorder=10, label='End')
    axes[1].set_title('BFGS Path on Rosenbrock Contours', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('y')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)


# Animated path
try: 
    mpl.rcParams['animation.embed_limit'] = 50  # MB - raise ceiling to 50 MB to see full animation

    if len(trace_params) > 1:
        # Increased hold_frames from 20 to 80 so the final minimum is clearly visible
        hold_frames = 80
        anim_path = np.vstack([trace_params, np.repeat([trace_params[-1]], hold_frames, axis=0)])
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis')
        line, = ax.plot([], [], color='red', linewidth=1.5)
        point, = ax.plot([], [], 'ro', markersize=4)
        # Static start/end markers so they are visible throughout the animation
        ax.scatter([trace_params[0, 0]], [trace_params[0, 1]], color='white', edgecolor='black', s=50, zorder=5, label='Start')
        ax.scatter([best_params[0]], [best_params[1]], color='yellow', edgecolor='black', s=50, zorder=10, label='End')
        ax.set_title('BFGS Path Animation', fontsize=12, fontweight='bold')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.legend()
        ax.grid(True, alpha=0.3)
        def init():
            line.set_data([], [])
            point.set_data([], [])
            return line, point
        def update(frame):
            xs = anim_path[:frame + 1, 0]
            ys = anim_path[:frame + 1, 1]
            line.set_data(xs, ys)
            point.set_data([anim_path[frame, 0]], [anim_path[frame, 1]])
            return line, point
        anim = animation.FuncAnimation(fig, update, init_func=init, frames=len(anim_path), interval=50, blit=True, repeat=False)
        display(HTML(anim.to_jshtml()))
        plt.close(fig)
        
except Exception as e:
    print('Inline animation rendering skipped:', e)

## Example: McCormick Function
Another classic optimization function is the McCormick function. With $f(x, y) = \sin(x + y) + (x - y)^2 - 1.5x + 2.5y + 1$. This has a minimum of -1.9133 at f(-0.54719, -1.54719).

Again we will run all three methods on this function and compare their results.


In [ ]:
def mccormick(params):
    x, y = params[0], params[1]
    return np.sin(x + y) + (x - y)**2 - 1.5*x + 2.5*y + 1

# Visualize McCormick function
x = np.linspace(-3, 4, 400)
y = np.linspace(-3, 4, 400)
X, Y = np.meshgrid(x, y)
Z = mccormick([X, Y])

fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(121)
contour = ax1.contour(X, Y, Z, levels=30, cmap='viridis')
ax1.plot(-0.54719, -1.54719, 'r*', markersize=20, label='Global Minimum (-0.54719, -1.54719)')
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_title('McCormick Function - Contour Plot', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8, edgecolor='none')
ax2.set_xlabel('x', fontsize=10)
ax2.set_ylabel('y', fontsize=10)
ax2.set_zlabel('f(x, y)', fontsize=10)
ax2.set_title('McCormick Function - 3D Surface', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Initialize optimization parameters
initial = Array[Double]([0.0, 0.0])
lower = Array[Double]([-3.0, -3.0])
upper = Array[Double]([4.0, 4.0])

mccormick_net = Func[Array[Double], Double](mccormick)

# Optimization routines
bfgs = BFGS(mccormick_net, 2, initial, lower, upper)
bfgs.Minimize()
bfgs_soln = bfgs.BestParameterSet.Values

nelder = NelderMead(mccormick_net, 2, initial, lower, upper)
nelder.Minimize()
nelder_soln = nelder.BestParameterSet.Values

powell = Powell(mccormick_net, 2, initial, lower, upper)
powell.Minimize()
powell_soln = powell.BestParameterSet.Values

print('\n' + ' Finding Minimum of McCormick Function:')

table = pd.DataFrame([
    {'Method': 'BFGS',
                'Result': np.round(bfgs_soln, 9),
                'Error': np.round(list(bfgs_soln) - np.array([ -0.547, -1.547 ]), 9)},
        {'Method': 'Nelder-Mead',
                'Result': np.round(nelder_soln, 9),
                'Error': np.round(list(nelder_soln) - np.array([ -0.547, -1.547 ]), 9)},
        {'Method': 'Powell',
                'Result': np.round(powell_soln, 9),
                'Error': np.round(list(powell_soln) - np.array([ -0.547, -1.547 ]), 9)}])

display(table)

## 4. Global Optimization
Global optimization methods are designed to find the global minimum across the entire search space, avoiding local minima. They typically require more function evaluations but are more robust for multimodal problems.

Methods Available
- Differential Evolution
- Multi-Start
- Multi-Start Single Linkage (MLSL)
- Particle Swarm
- Shuffled Complex Evolution
- Simulated Annealing

### Differential Evolution

Differential Evolution (DE) is a population-based evolutionary algorithm that's very robust for continuous optimization [[3]](#3). For each member $\mathbf{x}_i$ of the population, DE creates a trial vector $\mathbf{u}$ using mutation and crossover:

Mutation (DE/rand/1): Three distinct population members $\mathbf{x}_{r_0}$, $\mathbf{x}_{r_1}$, $\mathbf{x}_{r_2}$ are randomly selected, and a mutant vector is formed:

```math
\mathbf{v} = \mathbf{x}_{r_0} + G \cdot (\mathbf{x}_{r_1} - \mathbf{x}_{r_2})
```

where $G$ is a scale factor. 

Crossover (binomial): The trial vector $\mathbf{u}$ is assembled component-by-component:

```math
u_j = \begin{cases} v_j & \text{if } \text{rand}() \leq CR \text{ or } j = j_{\text{rand}} \\ x_{i,j} & \text{otherwise} \end{cases}
```

where $CR$ is the crossover probability and $j_{\text{rand}}$ is a randomly chosen index that ensures at least one component comes from the mutant vector.

Selection (greedy): The trial vector replaces the target only if it has equal or better fitness:

```math
\mathbf{x}_i^{(g+1)} = \begin{cases} \mathbf{u} & \text{if } f(\mathbf{u}) \leq f(\mathbf{x}_i^{(g)}) \\ \mathbf{x}_i^{(g)} & \text{otherwise} \end{cases}
```

The algorithm converges when the standard deviation of fitness values across the population falls below the tolerance.

Advantages: Very robust, handles discontinuities, good for difficult problems.

### Particle Swarm Optimization

PSO simulates social behavior of bird flocking or fish schooling [[4]](#4). Each particle $i$ has a position $\mathbf{x}_i$ and velocity $\mathbf{v}_i$ that are updated at each iteration using:

```math
v_{i,j}^{(t+1)} = w \cdot v_{i,j}^{(t)} + c_1 \cdot r_1 \cdot (p_{i,j} - x_{i,j}^{(t)}) + c_2 \cdot r_2 \cdot (g_j - x_{i,j}^{(t)})
```
```math
x_{i,j}^{(t+1)} = x_{i,j}^{(t)} + v_{i,j}^{(t+1)}
```
The three terms represent momentum (continue in the current direction), cognitive pull (return toward personal best), and social pull (move toward the swarm's best).

Advantages: Fast convergence, simple concept, works well for continuous problems.

### Shuffled Complex Evolution (SCE-UA)

SCE-UA was specifically developed for calibrating hydrological models [[5]](#5). The algorithm partitions the population into $p$ complexes, evolves each complex independently using a Competitive Complex Evolution (CCE) strategy, then shuffles members between complexes to share information.

The CCE step within each complex follows 4 steps: sub-complex selection, reflection, contraction, mutation.The shuffling step redistributes points across complexes, preventing any single complex from stagnating. This combination of local competitive evolution within complexes and global information sharing between complexes makes SCE-UA particularly effective for the rugged, multimodal objective functions typical of hydrological model calibration.

Advantages: Excellent for hydrological model calibration, balances exploration and exploitation.

### Simulated Annealing

SA mimics the physical process of annealing in metallurgy [[6]](#6). It accepts uphill moves with decreasing probability, allowing escape from local minima.

Advantages: Can escape local minima, works for discrete and continuous problems.

### Multi-Start Optimization

Combines local search with multiple random starting points.

Advantages: Simple to implement, leverages fast local search.       
Disadvantages: May waste evaluations in same basin of attraction.

### MLSL (Multi-Level Single Linkage)

Clustering-based global optimization that avoids redundant local searches.

Advantages: More efficient than multi-start, avoids redundant searches.

## Example: 5D Rosenbrock Function
Below we apply these to a 5D Rosenbrock objective function.


In [ ]:
# 5D objective visualized via 2D slice at fixed remaining coordinates.
lower = Array[Double]([-2.048, -2.048, -2.048, -2.048, -2.048])
upper = Array[Double]([2.048, 2.048, 2.048, 2.048, 2.048])

# Optimize with particle swarm
particle = ParticleSwarm(rosenbrock_net, 5, lower, upper)
particle.PopulationSize = 100
particle.MaxIterations = 100000
particle.Minimize()
particle_soln = particle.BestParameterSet.Values

# Optimize with simulated annealing
anneling = SimulatedAnnealing(rosenbrock_net, 5, lower, upper)
anneling.Minimize()
anneling_soln = anneling.BestParameterSet.Values

# Optimize with differential evolution
de = DifferentialEvolution(rosenbrock_net, 5, lower, upper)
de.PopulationSize = 100
de.MaxIterations = 5000
de.Minimize()
de_soln = de.BestParameterSet.Values

# Optimize with MultiStart
ms_initial = Array[Double]([0.0, 0.0, 0.0, 0.0, 0.0]) # Requires an intial guess with the same number of dimensions as the problem
ms = MultiStart(rosenbrock_net, 5, ms_initial, lower, upper, LocalMethod.BFGS) # Requires a local method to use for the local optimization steps
ms.MaxIterations = 50  # number of local starts
ms.Minimize()
ms_soln = ms.BestParameterSet.Values

# Optimize with MLSL
mlsl = MLSL(rosenbrock_net, 5, ms_initial, lower, upper, LocalMethod.BFGS) # Requires an itital guess with the same number of dimensions as the problem
mlsl.SampleSize = 30
mlsl.MaxIterations = 200
mlsl.Minimize()
mlsl_soln = mlsl.BestParameterSet.Values

# Optimize with Shuffled Complex Evolution
sce = ShuffledComplexEvolution(rosenbrock_net, 5, lower, upper)
sce.Complexes = 5
sce.MaxIterations = 2000
sce.Minimize()
sce_soln = sce.BestParameterSet.Values

print('\n'  + " Finding Minimum of Rosenbrock Function:")

table = pd.DataFrame([
    {'Method': 'Particle Swarm',
                'Result': np.round(particle_soln, 14),
                'Error': np.round(list(particle_soln) - np.array([1, 1, 1, 1, 1]), 14)},
        {'Method': 'Simulated Annealing',
                'Result': np.round(anneling_soln, 4),
                'Error': np.round(list(anneling_soln) - np.array([1, 1, 1, 1, 1]), 4)},
        {'Method': 'Differential Evolution',
                'Result': np.round(de_soln, 14),
                'Error': np.round(list(de_soln) - np.array([1, 1, 1, 1, 1]), 14)},
        {'Method': 'MultiStart',
                'Result': np.round(ms_soln, 14),
                'Error': np.round(list(ms_soln) - np.array([1, 1, 1, 1, 1]), 14)},
        {'Method': 'MLSL',
                'Result': np.round(mlsl_soln, 14),
                'Error': np.round(list(mlsl_soln) - np.array([1, 1, 1, 1, 1]), 14)},
        {'Method': 'Shuffled Complex Evolution',
                'Result': np.round(sce_soln, 14),
                'Error': np.round(list(sce_soln) - np.array([1, 1, 1, 1, 1]), 14)}
                ])

display(table)


## Examples: McCormick, Eggholder, Ackley, Three Hump Camel
We continue with the McCormick example from the local optimization section. We also introduce other test functions for optimization that are a little more complicated.

Eggholder: $f(x, y) = -(y + 47)\sin\left(\sqrt{|x/2 + (y + 47)|}\right) - x \cdot \sin\left(\sqrt{|x - (y + 47)|}\right)$ with a minimum of -959.6407 at $f(512, 404.2319)$   

Ackley: $f(x, y) = -20 \cdot \exp[-0.2\sqrt{0.5(x^2 + y^2)}] - \exp[0.5(\cos(2\pi x) + \cos(2\pi y))] + e + 20$ with a minimum of 0 at $f(0, 0)$  
      
Three Hump Camel: $f(x, y) = 2x^2 - 1.05x^4 + x^6/6 + xy + y^2$ with a minimum of 0 at $f(0, 0)$


In [ ]:
def egghold(params):
    x, y = params[0], params[1]
    return -(y + 47) * np.sin(np.sqrt(abs(x/2 + (y + 47)))) - x * np.sin(np.sqrt(abs(x - (y + 47))))

def ackley(params):
    x, y = params[0], params[1]
    return -20 * np.exp(-0.2 * np.sqrt(0.5 * (x*x + y*y))) - np.exp(0.5 * (np.cos(2*np.pi*x) + np.cos(2*np.pi*y))) + np.e + 20

def three_hump_camel(params):
    x, y = params[0], params[1]
    return 2*x*x - 1.05*x**4 + (x**6)/6 + x*y + y*y

def run_global_benchmark(func, name, lower_b, upper_b, min_x, min_y):
    # Visualize
    x = np.linspace(lower_b, upper_b, 400)
    y = np.linspace(lower_b, upper_b, 400)
    X, Y = np.meshgrid(x, y)
    Z = func([X, Y])

    fig = plt.figure(figsize=(14, 5))
    ax1 = fig.add_subplot(121)
    contour = ax1.contour(X, Y, Z, levels=30, cmap='viridis')
    ax1.plot(min_x, min_y, 'r*', markersize=20, label=f'Global Minimum ({min_x}, {min_y})')
    ax1.set_xlabel('x', fontsize=12)
    ax1.set_ylabel('y', fontsize=12)
    ax1.set_title(f'{name} - Contour Plot', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(122, projection='3d')
    ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8, edgecolor='none')
    ax2.set_xlabel('x', fontsize=10)
    ax2.set_ylabel('y', fontsize=10)
    ax2.set_zlabel('f(x, y)', fontsize=10)
    ax2.set_title(f'{name} - 3D Surface', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    lower = Array[Double]([lower_b, lower_b])
    upper = Array[Double]([upper_b, upper_b])

    func_net = Func[Array[Double], Double](func)

    # Optimize with particle swarm
    particle = ParticleSwarm(func_net, 2, lower, upper)
    particle.PopulationSize = 100
    particle.MaxIterations = 100000
    particle.Minimize()
    particle_soln = particle.BestParameterSet.Values

    # Optimize with simulated annealing
    anneling = SimulatedAnnealing(func_net, 2, lower, upper)
    anneling.Minimize()
    anneling_soln = anneling.BestParameterSet.Values

    # Optimize with differential evolution
    de = DifferentialEvolution(func_net, 2, lower, upper)
    de.PopulationSize = 100
    de.MaxIterations = 5000
    de.Minimize()
    de_soln = de.BestParameterSet.Values

    # Optimize with MultiStart
    ms_initial = Array[Double]([0.0, 0.0])
    ms = MultiStart(func_net, 2, ms_initial, lower, upper, LocalMethod.BFGS)
    ms.MaxIterations = 50
    ms.Minimize()
    ms_soln = ms.BestParameterSet.Values

    # Optimize with MLSL
    mlsl = MLSL(func_net, 2, ms_initial, lower, upper, LocalMethod.BFGS)
    mlsl.SampleSize = 30
    mlsl.MaxIterations = 200
    mlsl.Minimize()
    mlsl_soln = mlsl.BestParameterSet.Values

    # Optimize with Shuffled Complex Evolution
    sce = ShuffledComplexEvolution(func_net, 2, lower, upper)
    sce.Complexes = 5
    sce.MaxIterations = 2000
    sce.Minimize()
    sce_soln = sce.BestParameterSet.Values

    print('\n' + f' Finding Minimum of {name}:')
    table = pd.DataFrame([
        {'Method': 'Particle Swarm',
                    'Result': np.round(particle_soln, 6)},
            {'Method': 'Simulated Annealing',
                    'Result': np.round(anneling_soln, 6)},
            {'Method': 'Differential Evolution',
                    'Result': np.round(de_soln, 6)},
            {'Method': 'MultiStart',
                    'Result': np.round(ms_soln, 6)},
            {'Method': 'MLSL',
                    'Result': np.round(mlsl_soln, 6)},
            {'Method': 'Shuffled Complex Evolution',
                    'Result': np.round(sce_soln, 6)}
                    ])
    display(table)

# Run global benchmarks
run_global_benchmark(mccormick, 'McCormick Function', -3.0, 4.0, -0.54719, -1.54719)
run_global_benchmark(egghold, 'Egghold Function', -512.0, 512.0, 512, 404.2319)
run_global_benchmark(ackley, 'Ackley Function', -5.0, 5.0, 0, 0)
run_global_benchmark(three_hump_camel, 'Three Hump Camel Function', -5.0, 5.0, 0, 0)


Sticking with the Eggholder function we will take a look at the path the Differential Evolition solver takes when trying to find the minimum!


In [ ]:
eggholder_net = Func[Array[Double], Double](egghold)
lower = Array[Double]([-512, -512])
upper = Array[Double]([512, 512])

# Contour grid
x_eg = np.linspace(-512, 512, 400)
y_eg = np.linspace(-512, 512, 400)
X_eg, Y_eg = np.meshgrid(x_eg, y_eg)
Z_eg = egghold([X_eg, Y_eg])

# Run DE with trace recording
de_trace = DifferentialEvolution(eggholder_net, 2, lower, upper)
de_trace.PopulationSize = 100
de_trace.MaxIterations = 5000
de_trace.RecordTraces = True
de_trace.Minimize()

# Extract trace
trace_list = list(de_trace.ParameterSetTrace)
trace_params = np.array([[float(ps.Values[0]), float(ps.Values[1])] for ps in trace_list], dtype=float)
trace_values = np.array([float(ps.Fitness) for ps in trace_list], dtype=float)

# Ensure best parameter set is included at the end
best_params = np.array([float(de_trace.BestParameterSet.Values[0]), float(de_trace.BestParameterSet.Values[1])], dtype=float)
if len(trace_params) == 0 or not np.allclose(trace_params[-1], best_params):
    trace_params = np.vstack([trace_params, best_params])
    trace_values = np.append(trace_values, float(de_trace.BestParameterSet.Fitness))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Objective trace
axes[0].plot(trace_values, color='steelblue', linewidth=1)
axes[0].set_title('DE Objective Trace (Eggholder)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Objective value')
axes[0].grid(True, alpha=0.3)

# Contour with path
if len(trace_params) > 1:
    axes[1].contour(X_eg, Y_eg, Z_eg, levels=30, cmap='viridis')
    axes[1].plot(trace_params[:, 0], trace_params[:, 1], color='red', linewidth=2, alpha=0.7, label='DE path')
    axes[1].scatter([trace_params[0, 0]], [trace_params[0, 1]], color='white', edgecolor='black', s=50, zorder=5, label='Start')
    axes[1].scatter([best_params[0]], [best_params[1]], color='yellow', edgecolor='black', s=50, zorder=10, label='End')
    axes[1].set_title('DE Path on Eggholder Contours', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('y')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Animation
try:
    mpl.rcParams['animation.embed_limit'] = 80  # raise ceiling to 80 MB

    if len(trace_params) > 1:
        hold_frames = 80
        anim_path = np.vstack([trace_params, np.repeat([trace_params[-1]], hold_frames, axis=0)])
        # Skip every other frame
        anim_path = anim_path[::2]

        fig, ax = plt.subplots(figsize=(6, 5), dpi=72)
        ax.contour(X_eg, Y_eg, Z_eg, levels=30, cmap='viridis')
        line, = ax.plot([], [], color='red', linewidth=2)
        point, = ax.plot([], [], 'ro', markersize=4)

        # Static markers visible throughout
        ax.scatter([trace_params[0, 0]], [trace_params[0, 1]], color='white', edgecolor='black', s=50, zorder=5, label='Start')
        ax.scatter([best_params[0]], [best_params[1]], color='yellow', edgecolor='black', s=50, zorder=10, label='End')

        ax.set_title('DE Path Animation (Eggholder)', fontsize=12, fontweight='bold')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        def init():
            line.set_data([], [])
            point.set_data([], [])
            return line, point

        def update(frame):
            xs = anim_path[:frame + 1, 0]
            ys = anim_path[:frame + 1, 1]
            line.set_data(xs, ys)
            point.set_data([anim_path[frame, 0]], [anim_path[frame, 1]])
            return line, point

        anim = animation.FuncAnimation(fig, update, init_func=init, frames=len(anim_path),
                                        interval=50, blit=True, repeat=False)
        display(HTML(anim.to_jshtml()))
        plt.close(fig)

except Exception as e:
    print('Inline animation rendering skipped:', e)

## Constrained Optimization (Augmented Lagrange)

We will now use the Rosenbrock function constrained to a disk to test the Augmented Lagrange Algorithm, which handles equality and inequality constraints by adding penalty terms to the objective function [[7]](#7). This method uses an inner solver; here we give it BFGS, which is gradient-based. The constraint handling is separate from the inner solver, so you can swap in derivative-free solvers when needed.

**Constraint Types**:
- `ConstraintType.EqualTo`: Equality constraint $g(\mathbf{x}) = 0$
- `ConstraintType.LesserThanOrEqualTo`: Inequality constraint $g(\mathbf{x}) \leq 0$
- `ConstraintType.GreaterThanOrEqualTo`: Inequality constraint $g(\mathbf{x}) \geq 0$



In [ ]:
def constraint(x):
    return x[0]*x[0] + x[1]*x[1]

def rosenbrock_disk(params):
    return np.power(1 - params[0], 2) + 100*(np.power(params[1] - params[0]*params[0], 2))

rosenbrock_disk_net = Func[Array[Double], Double](rosenbrock_disk)
constraint_net = Func[Array[Double], Double](constraint)

# Set up inner solver
initial = Array[Double]([0.0, 0.0])
lower = Array[Double]([-1.5, -1.5])
upper = Array[Double]([1.5, 1.5])
inner_solver = BFGS(rosenbrock_disk_net, 2, initial, lower, upper)

# Set up constraint
constraint = Constraint(constraint_net, 2, 2, ConstraintType.LesserThanOrEqualTo)

# Solve
constraint_list = List[IConstraint]()
constraint_list.Add(constraint)

solver = AugmentedLagrange(rosenbrock_disk_net, inner_solver, constraint_list)
solver.Minimize()
soln = solver.BestParameterSet.Values

# Results
print('\n'  + " Finding Minimum of Rosenbrock Function Constrained to a Disk:")

table = pd.DataFrame([
    {'Method': 'Augmented Lagrange',
                'Result': np.round(soln, 6),
                'Error': np.round(list(soln) - np.array([1, 1]), 6)},
])
display(table)


## When to Use Each Method
|Category|Method|Best Use|
|---|---|---|
|Local Optimization | BFGS | Smooth objectives, fast convergence |
|Local Optimization | Nelder-Mead | Noisy/derivative-free |
|Local Optimization | Powell | Derivative-free, good for moderate dimensions |
|Global Optimization | Particle Swarm | Multimodal, easy tuning |
|Global Optimization | Differential Evolution (DE) | Robust global search |
|Global Optimization | Simulated Annealing | Escapes local minima, simple setup |
|Global Optimization | SCE (Shuffled Complex Evolution) | Hydrologic calibration |
|Constrained | Augmented Lagrange | Explicit constraints |


### Practical Tips
- Start with broad bounds to avoid excluding the true optimum
- Use multiple starts for non-convex objectives
- For noisy objectives, prefer derivative-free methods (Nelder-Mead, Powell)
- Always verify solutions by evaluating the objective value at the returned parameters

## Exercise
1. Fit Rosenbrock in 3D with BFGS and DE
2. Compare runtime and final error
3. Try narrower bounds and observe convergence

## References

<a id="1">[1]</a> J. Nocedal and S. J. Wright, *Numerical Optimization*, 2nd ed. Springer, 2006.

<a id="2">[2]</a> J. A. Nelder and R. Mead, "A simplex method for function minimization," *The Computer Journal*, vol. 7, no. 4, pp. 308-313, 1965.

<a id="3">[3]</a> R. Storn and K. Price, "Differential evolution - a simple and efficient heuristic for global optimization over continuous spaces," *Journal of Global Optimization*, vol. 11, no. 4, pp. 341-359, 1997.

<a id="4">[4]</a> J. Kennedy and R. Eberhart, "Particle swarm optimization," *Proceedings of IEEE International Conference on Neural Networks*, vol. 4, pp. 1942-1948, 1995.

<a id="5">[5]</a> Q. Duan, S. Sorooshian, and V. K. Gupta, "Optimal use of the SCE-UA global optimization method for calibrating watershed models," *Journal of Hydrology*, vol. 158, no. 3-4, pp. 265-284, 1994.

<a id="6">[6]</a> S. Kirkpatrick, C. D. Gelatt, and M. P. Vecchi, "Optimization by simulated annealing," *Science*, vol. 220, no. 4598, pp. 671-680, 1983.

<a id="7">[7]</a> E. G. Birgin and J. M. Martinez, *Practical Augmented Lagrangian Methods for Constrained Optimization*. SIAM, 2014.